# PM2.5 Prediction Model

**Input:** `df_model_monthly.csv` built in `features.ipynb`.

| Model | Algorithm | Features                                              | Goal |
|---|---|-------------------------------------------------------|---|
| **A** | RF + XGBoost | environmental, spatial, contextual variables          | Policy story: what can municipalities act on? |
| **B** | RF | PM2.5 lags, rolling features, lagged pollutants, weather, context | Accuracy benchmark |
| **C** | Ridge | Same as A                                             | Linear baseline with signed coefficients |



## Import Libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.base import clone
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
print("Libraries loaded")


## Load final modeling dataset

This CSV was created in `features.ipynb` and already contains:
- date
- station identifier (`eoi_code`)
- weather/context variables
- lag features
- rolling features

Let's inspect it first to ensure everything is in order

In [ ]:
df = pd.read_csv("df_model_monthly.csv", parse_dates=["date"])

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Stations:", df["eoi_code"].nunique())

display(df.head())

In [ ]:
# Ensuring target exists, date is datetime, station id exists, PM2.5 has non-missing values
print("Target missing:", df["PM2_5"].isna().sum())
print("Date dtype:", df["date"].dtype)
print("Station id missing:", df["eoi_code"].isna().sum())

display(df[["date", "eoi_code", "PM2_5"]].head())

## Filtering dataset

Before building individual models, we create one shared filtered dataset.

This keeps Models A, B, and C comparable:
- same rows
- same target
- same station grouping
- same time split
- same spatial split

Model B requires history features for the forecasting, such as lag values and mean across 3 past months. We'll be filtering only on PM2.5 history, we do not filter on pollutant lags like O3, because those reflect real sensor availability and would remove too many observations.

In [ ]:
required_history = [
    "PM2_5_lag1",
    "PM2_5_lag2",
    "PM2_5_lag3",
    "PM2_5_roll3_mean",
    "PM2_5_roll3_std"
]

df_model = df.dropna(subset=required_history).copy()

print("Rows before filtering:", len(df))
print("Rows after common history filter:", len(df_model))
print("Stations after filter:", df_model["eoi_code"].nunique())

display(df_model.head())

## Creating target and grouping variable

- `y` = PM2.5 target
- `groups` = station identifier for spatial split

In [ ]:
target_col = "PM2_5"
y = df_model[target_col].copy()
groups = df_model["eoi_code"].copy()

print("Filtered modeling shape:", df_model.shape)
print("Target shape:", y.shape)
print("Unique stations:", groups.nunique())

## Common train/test splits

We define the split and reuse it for all models.

### Time split
Train on earlier months, test on later months

In [ ]:
# Time split
cutoff_date = df_model["date"].quantile(0.8)
time_train_mask = df_model["date"] <= cutoff_date
time_test_mask = df_model["date"] > cutoff_date

print("Cutoff:", cutoff_date.date())
print("Train/test:", time_train_mask.sum(), time_test_mask.sum())

### Spatial split
Train on some stations, test on unseen stations

In [ ]:
# Spatial split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(splitter.split(df_model, y, groups=groups))

print("Spatial train/test rows:", len(tr_idx), len(te_idx))
print("Train stations:", df_model.iloc[tr_idx]["eoi_code"].nunique())
print("Test stations:", df_model.iloc[te_idx]["eoi_code"].nunique())

## Model A  (Random Forest & XGBoost)

In this section, we build Model A, an explanatory model designed to understand how environmental, spatial, and contextual factors influence PM2.5 levels.

Unlike predictive models that rely on pollutant inputs, Model A uses only non-pollutant features, allowing us to interpret the underlying drivers of air pollution.


### Modeling Approach

We train two models:

- Random Forest (RF)
- XGBoost (XGB)

Both models are evaluated under two scenarios:

- Time split - forecasting future observations
- Spatial split - generalizing to unseen locations


In [ ]:
# Model A feature set
# If a column name does not exist in df_model, it will be skipped safely.

candidate_features_a = [
    # Seasonality
    "Season",
    "month_sin",
    "month_cos",

    # Weather
    "Temp_Mean",
    "Wind_Speed",
    "Precipitation",

    # Lagged weather
    "Temp_Mean_lag1",
    "Wind_Speed_lag1",
    "Precipitation_lag1",

    # Spatial
    "Altitude",
    "Latitude",
    "Longitude",
    "Green_Ratio",
    "Population_Density",

    # Contextual / station data
    "Station_Type",
    "Station_Area",
]

features_a = [c for c in candidate_features_a if c in df_model.columns]

print("Model A features:", len(features_a))
print(features_a)


## Preprocessing and Evaluation Setup

We define reusable functions for data preparation and model evaluation.

**Preprocessing**:
  - Numerical features - median imputation + standard scaling
  - Categorical features - most frequent imputation + one-hot encoding

**Evaluation metrics**:
  - **MAE** — average error
  - **RMSE** — penalizes larger errors
  - **R²** — explained variance

These will be used across all models.

In [ ]:
def make_preprocessor(feature_list):
    num_cols = [c for c in feature_list if pd.api.types.is_numeric_dtype(df_model[c])]
    cat_cols = [c for c in feature_list if c not in num_cols]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot",  OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])

def reg_metrics(y_true, y_pred):
    return {
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2":   r2_score(y_true, y_pred),
    }

pre_a = make_preprocessor(features_a)
print("Preprocessors and metrics helper ready")

### Build Model A feature matrix

We now create the feature matrix for Model A only.

Important:
- `y` is already defined
- split masks and indices are already defined
- only `X_a` is model-specific

In [ ]:
X_a = df_model[features_a].copy()

print("Model A X shape:", X_a.shape)
display(X_a.head())

# y, groups, time_train_mask, time_test_mask, tr_idx, te_idx
# were already created in the setup section above.

### Model A — Time split

The dataset is divided based on the date:
- the model is trained on **earlier months**
- the model is tested on **later months**

This simulates a realistic forecasting scenario, where we use past data to predict future air pollution levels.

We train both:
- Random Forest (RF)
- XGBoost (XGB)

and compare their performance using MAE, RMSE, and R².

In [ ]:
# Time split data
X_train_a_t = X_a.loc[time_train_mask] # Features for training(past data)
X_test_a_t = X_a.loc[time_test_mask] # Features for testing (future data)
y_train_t = y.loc[time_train_mask] # values for training
y_test_t = y.loc[time_test_mask] # values for testing

# RF - time split
# Build Pipeline: Preprocessing + RF

rf_a_t = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )),
])

# Train model on past data
rf_a_t.fit(X_train_a_t, y_train_t)
pred_rf_a_t = rf_a_t.predict(X_test_a_t) # Predict PM2.5 on future data
m_rf_a_t = reg_metrics(y_test_t, pred_rf_a_t) # Evaluate predictions using MAE, RMSE, and R2
print("RF time split:", m_rf_a_t)

# XGBoost — time split
# Same process as RF
xgb_a_t = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )),
])
xgb_a_t.fit(X_train_a_t, y_train_t)
pred_xgb_a_t = xgb_a_t.predict(X_test_a_t)
m_xgb_a_t = reg_metrics(y_test_t, pred_xgb_a_t)

print("XGB time split:", m_xgb_a_t)

### Model A — Spatial split

Instead of splitting the data by time, we split it by **monitoring stations** (`eoi_code`).

- the model is trained on data from a subset of stations
- the model is tested on **completely unseen stations**

We train both:
- Random Forest(RF)
- XGBoost(XGB)

In [ ]:
# Spatial split data
# Split the dataset using indices created earlier (GroupShuffleSplit)
# Ensures that train and test contain DIFFERENT stations

X_train_a_s = X_a.iloc[tr_idx] # Features for training (some stations)
X_test_a_s = X_a.iloc[te_idx] # Features for testing (unseen stations)
y_train_s = y.iloc[tr_idx] # values for training
y_test_s = y.iloc[te_idx] # values for testing

# RF — spatial split
rf_a_s = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )),
])
rf_a_s.fit(X_train_a_s, y_train_s)
pred_rf_a_s = rf_a_s.predict(X_test_a_s)
m_rf_a_s = reg_metrics(y_test_s, pred_rf_a_s)
print("RF spatial split:", m_rf_a_s)

# XGBoost — spatial split
xgb_a_s = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )),
])
xgb_a_s.fit(X_train_a_s, y_train_s)
pred_xgb_a_s = xgb_a_s.predict(X_test_a_s)
m_xgb_a_s = reg_metrics(y_test_s, pred_xgb_a_s)
print("XGB spatial split:", m_xgb_a_s)


### Model A — Model Selection

The final model (Champion A) is selected based on:

> Lowest average RMSE across both time and spatial splits

This ensures the chosen model performs well both temporally (future prediction) and spatially (new locations)

The selected model is retrained on the full dataset to:
- maximize available information
- enable interpretation via SHAP
- support feature importance analysis


In [ ]:
# Champion selection by lowest avg RMSE across both splits
avg_rmse_rf = (m_rf_a_t["RMSE"] + m_rf_a_s["RMSE"]) / 2
avg_rmse_xgb = (m_xgb_a_t["RMSE"] + m_xgb_a_s["RMSE"]) / 2

print(f"RF avg RMSE: {avg_rmse_rf:.4f}")
print(f"XGB avg RMSE: {avg_rmse_xgb:.4f}")

if avg_rmse_rf <= avg_rmse_xgb:
    champion_a_name = "RF"
    pred_champ_a_t = pred_rf_a_t
    pred_champ_a_s = pred_rf_a_s
    m_champ_a_t = m_rf_a_t
    m_champ_a_s = m_rf_a_s
else:
    champion_a_name = "XGBoost"
    pred_champ_a_t = pred_xgb_a_t
    pred_champ_a_s = pred_xgb_a_s
    m_champ_a_t = m_xgb_a_t
    m_champ_a_s = m_xgb_a_s

print(f"\nChampion A: {champion_a_name}")
print(f"Time split - RMSE={m_champ_a_t['RMSE']:.4f}  MAE={m_champ_a_t['MAE']:.4f}  R2={m_champ_a_t['R2']:.4f}")
print(f"Spatial split - RMSE={m_champ_a_s['RMSE']:.4f}  MAE={m_champ_a_s['MAE']:.4f}  R2={m_champ_a_s['R2']:.4f}")

# Retrain champion on FULL dataset (for SHAP / feature importance)
if champion_a_name == "RF":
    champ_a_full = Pipeline([
        ("preprocess", clone(pre_a)),
        ("model", RandomForestRegressor(
            n_estimators=400, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )),
    ])
else:
    champ_a_full = Pipeline([
        ("preprocess", clone(pre_a)),
        ("model", XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=5,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0
        )),
    ])

champ_a_full.fit(X_a, y)
print(f"\nchamp_a_full retrained on full dataset ({len(X_a)} rows).")


### Model A — Summary table

In [ ]:
metrics_a = pd.DataFrame([
    {"split": "time", "model": f"RF_A", **m_rf_a_t},
    {"split": "spatial", "model": f"RF_A", **m_rf_a_s},
    {"split": "time", "model": f"XGBoost_A", **m_xgb_a_t},
    {"split": "spatial", "model": f"XGBoost_A", **m_xgb_a_s},
])
display(metrics_a)
metrics_a.to_csv("model_output/model_a_metrics.csv", index=False)
print(f"Champion: {champion_a_name}")


**Random Forest (RF)** slightly outperforms XGBoost on average RMSE across 2 splits

RF shows:
  - slightly higher error on time split
  - but **better generalization on spatial split**

So Random Forest is chosen as **Champion A**

The model is then **retrained on the full dataset** to:
- use all available information
- enable interpretation (SHAP analysis)
- support feature importance and policy insights

This champion model will be used for all subsequent comparison.

### Model A — Performance Comparison (RF vs XGBoost)

In [ ]:
plot_a = metrics_a.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_a,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title="Model A — RMSE and MAE by split",
)
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modela_matrix_comparison.png", scale=2)
fig.show()


### Model A — Residuals

In [ ]:
# residuals
r_rf_a_s = y_test_s.values - pred_rf_a_s
r_xgb_a_s = y_test_s.values - pred_xgb_a_s

# common bins so both plots are comparable
all_r_s = np.concatenate([r_rf_a_s, r_xgb_a_s])
bins_s = np.histogram_bin_edges(all_r_s, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Model A RF residuals (spatial test)", "Model A XGBoost residuals (spatial test)")
)

fig.add_trace(
    go.Histogram(
        x=r_rf_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="RF",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_xgb_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="XGB",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title="Model A residuals (spatial test)",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_residuals_space_rf_xgb.png", scale=2)
fig.show()

In [ ]:
# residuals
r_rf_a_t = y_test_t.values - pred_rf_a_t
r_xgb_a_t = y_test_t.values - pred_xgb_a_t

# common bins so both plots are comparable
all_r_t = np.concatenate([r_rf_a_t, r_xgb_a_t])
bins_t = np.histogram_bin_edges(all_r_t, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Model A RF residuals (time test)", "Model A XGBoost residuals (time test)")
)

fig.add_trace(
    go.Histogram(
        x=r_rf_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="RF",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_xgb_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="XGB",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title="Model A residuals (time test)",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_residuals_time_rf_xgb.png", scale=2)
fig.show()

The charts highlights how close the two models perform across both splits.

Overall, the differences in **RMSE and MAE are very small**, indicating that:
- Random Forest and XGBoost perform almost identically
- neither model clearly dominates across all settings

The models essentially go head-to-head, with only marginal differences in error.

The final selection is based on **slight differences in average performance**, rather than a large gap in predictive power.

### Model A — SHAP (Champion RF)

To understand *why* Model A predicts what it does — not just *how well* — we apply SHAP to the champion Random Forest model.

This section covers:
- Computing mean absolute SHAP values for each feature to rank their overall importance
- Visualising a **beeswarm plot** to see how feature values (low - high) push predictions up or down
- Visualising a **bar chart** of the top 15 features by mean SHAP value

The goal is to move beyond a black-box model and identify which environmental, spatial, and contextual variables actually drive PM2.5 predictions.

In [ ]:
import scipy.sparse as sp
import shap

# use the already retrained full champion model
Xp = champ_a_full.named_steps["preprocess"].transform(X_a)

if sp.issparse(Xp):
    Xp = Xp.toarray()

Xp = np.asarray(Xp, dtype=np.float64)

# sample to make SHAP faster
rng = np.random.RandomState(42)
idx = rng.choice(Xp.shape[0], size=min(800, Xp.shape[0]), replace=False)
Xs = Xp[idx]

# works for both RF and XGBoost
explainer = shap.TreeExplainer(champ_a_full.named_steps["model"])
sv = explainer.shap_values(Xs)

fn = champ_a_full.named_steps["preprocess"].get_feature_names_out()

# global SHAP importance
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance_a = (
    pd.DataFrame({
        "feature": fn,
        "mean_abs_shap": mean_abs_shap
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

shap_importance_a.to_csv(f"model_output/modela_{champion_a_name}_shap_importance.csv", index=False)
display(shap_importance_a.head(15))

In [ ]:
# Plotly view
top_n = 15
top_features = shap_importance_a.head(top_n)["feature"].tolist()
feat_to_rank = {f: i for i, f in enumerate(top_features)}

long_df = pd.DataFrame({
    "feature": np.repeat(fn, Xs.shape[0]),
    "shap_value": sv.T.flatten(),
    "feature_value": Xs.T.flatten(),
})

long_df = long_df[long_df["feature"].isin(top_features)].copy()
long_df["feature_rank"] = long_df["feature"].map(feat_to_rank)

long_df["feature_value_norm"] = long_df.groupby("feature")["feature_value"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)
long_df["feature_value_norm"] = long_df["feature_value_norm"].clip(-2.5, 2.5)

jitter_rng = np.random.RandomState(42)
long_df["y_jitter"] = long_df["feature_rank"] + jitter_rng.uniform(-0.28, 0.28, size=len(long_df))

fig_swarm_a = px.scatter(
    long_df,
    x="shap_value",
    y="y_jitter",
    color="feature_value_norm",
    color_continuous_scale=[
        [0.0, "#1f6bff"],
        [1.0, "#ff4fa3"],
    ],
    range_color=[-2.5, 2.5],
    opacity=0.75,
    title=f"SHAP — Model A Champion ({champion_a_name}): per-point beeswarm direction view (top {top_n} features)",
    labels={
        "shap_value": "SHAP value (impact on PM2.5)",
        "feature_value_norm": "feature value (z-score)"
    },
    hover_data={
        "feature": True,
        "feature_value": ':.3f',
        "feature_value_norm": ':.3f',
        "shap_value": ':.3f',
        "y_jitter": False
    },
)

fig_swarm_a.update_traces(marker={"size": 6})
fig_swarm_a.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(
        title="",
        tickmode="array",
        tickvals=list(range(top_n)),
        ticktext=top_features,
        autorange="reversed",
    ),
)

fig_swarm_a.update_coloraxes(
    colorbar_title="Feature value<br>(low → high)"
)

fig_swarm_a.add_vline(x=0, line_dash="dash", line_color="black")
fig_swarm_a.write_image(f"images/modela_{champion_a_name}_shap_beeswarm.png", scale=2)
fig_swarm_a.show()

**How the color scale is formed (SHAP beeswarm)**

Colors show feature value level, not SHAP magnitude.
For each feature, values are standardized with:

$$z = \frac{x - \mu}{\sigma + 10^{-9}}$$

So:

- Blue = lower-than-average value (negative (z))
- Pink = higher-than-average value (positive (z))

This makes colors comparable across features with different units.

In [ ]:
top_n = 15

top_feats_a = shap_importance_a.head(top_n).copy().iloc[::-1]

fig_imp_a = px.bar(
    top_feats_a,
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title=f"Model A Champion ({champion_a_name}) — top {top_n} features by SHAP importance",
    labels={
        "mean_abs_shap": "mean |SHAP value|",
        "feature": "feature"
    },
)

fig_imp_a.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)"
)

fig_imp_a.write_image(f"images/modela_{champion_a_name}_shap_importance_bar.png", scale=2)
fig_imp_a.show()


#### Interpretation

SHAP was chosen over built-in feature importances because it provides both **magnitude** (how much a feature moves the prediction) and **direction** (does a higher value push PM2.5 up or down?), which is essential for translating model outputs into actionable policy.

**Key findings from the top 15 features:**

- `month_cos` is the single most influential feature (mean |SHAP| ≈ 4.54), capturing the strong winter–summer seasonality of PM2.5 in Central Italy — cold months consistently drive concentrations up.
- `Latitude` and `Longitude` rank second and third, confirming that geography plays a major structural role: northern and inland stations tend to record higher PM2.5 regardless of weather conditions.
- `Temp_Mean` (rank 4) has a negative direction — higher temperatures are associated with lower PM2.5 — consistent with summer dispersion and reduced heating emissions.
- `Altitude` (rank 5) acts as a natural buffer: higher-elevation stations show systematically lower PM2.5, as expected from reduced traffic and better atmospheric mixing.
- Weather dispersion variables (`Precipitation`, `Wind_Speed`, and their lag-1 counterparts) appear in the mid-range, confirming that rainfall and wind reduce particulate accumulation — with lagged effects persisting into the following month.
- `Green_Ratio` and `Population_Density` contribute meaningfully, supporting the use of urban structure as a policy lever.
- Station area type (`Rural` / `Suburban`) captures local exposure context that the continuous variables alone cannot explain.

The beeswarm plot further confirms that most features have well-behaved, monotonic SHAP distributions — high feature values push in one consistent direction — giving confidence that the model has learned interpretable real-world relationships rather than noise.

## Model B — Random Forest (Environmental + PM2.5 History + Lagged Pollutants)

Accuracy benchmark. Adds PM2.5 lags, rolling stats, and lagged co-pollutants on top of all Model A features.
Comparing A vs B shows how much pollution memory and co-pollutant history improves forecasting.

In [ ]:
# Model B feature set
features_b = [
    "PM2_5_lag1", "PM2_5_lag2", "PM2_5_lag3", "PM2_5_roll3_mean", "PM2_5_roll3_std",
    "PM10_lag1", "NO2_lag1", "O3_lag1",
    "Temp_Mean", "Wind_Speed", "Precipitation",
    "Temp_Mean_lag1", "Wind_Speed_lag1", "Precipitation_lag1",
    "month_sin", "month_cos",
    "Altitude", "Latitude", "Longitude",
    "Green_Ratio", "Population_Density",
    "Station_Type", "Station_Area",
]
features_b = [c for c in features_b if c in df_model.columns]
X_b = df_model[features_b].copy()

print("Model B features:", len(features_b))
print(features_b)
print("Model B X shape:", X_b.shape)

### Model B — Time split

In [ ]:
def make_preprocessor_b(feature_list):
    num_cols_b = [c for c in feature_list if pd.api.types.is_numeric_dtype(df_model[c])]
    cat_cols_b = [c for c in feature_list if c not in num_cols_b]
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols_b),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols_b),
    ])

pre_b = make_preprocessor_b(features_b)

# Time split
X_train_b_t = X_b.loc[time_train_mask]
X_test_b_t = X_b.loc[time_test_mask]
y_train_t = y.loc[time_train_mask]
y_test_t = y.loc[time_test_mask]

rf_b = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b.fit(X_train_b_t, y_train_t)
pred_rf_b_t = rf_b.predict(X_test_b_t)
m_rf_b_t = reg_metrics(y_test_t, pred_rf_b_t)
print("Model B time split:", m_rf_b_t)

### Model B — Spatial split

In [ ]:
X_train_b_s = X_b.iloc[tr_idx]
X_test_b_s = X_b.iloc[te_idx]
y_train_s = y.iloc[tr_idx]
y_test_s = y.iloc[te_idx]

rf_b_s = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b_s.fit(X_train_b_s, y_train_s)
pred_rf_b_s = rf_b_s.predict(X_test_b_s)
m_rf_b_s = reg_metrics(y_test_s, pred_rf_b_s)
print("Model B spatial split:", m_rf_b_s)

In [ ]:
metrics_b = pd.DataFrame([
    {"split": "time",    "model": "RandomForest_B", **m_rf_b_t},
    {"split": "spatial", "model": "RandomForest_B", **m_rf_b_s},
])
display(metrics_b)
metrics_b.to_csv("model_output/modelb_results.csv", index=False)
print("Saved to model_output/modelb_results.csv")

In [ ]:
plot_b = metrics_b.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_b,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title="Model B — RMSE and MAE by split",
)
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modelb_metrics.png", scale=2)
fig.show()


In [ ]:
r = y_test_t.values - pred_rf_b_t
fig = px.histogram(
    x=r,
    nbins=30,
    title="Model B residuals (time test)",
    labels={"x": "actual - pred", "y": "count"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.write_image("images/modelb_residuals_time.png", scale=2)
fig.show()


In [ ]:
r = y_test_s.values - pred_rf_b_s
fig = px.histogram(
    x=r,
    nbins=30,
    title="Model B residuals (spatial test)",
    labels={"x": "actual - pred", "y": "count"},
)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig.write_image("images/modelb_residuals_space.png", scale=2)
fig.show()


### SHAP test 

SHAP helps us explain **why** Model B predicts higher or lower PM2.5 for each data point.

How it works:
- For each row, SHAP assigns a contribution value to every feature.
- A **positive SHAP value** means that feature pushes the prediction **up**.
- A **negative SHAP value** means that feature pushes the prediction **down**.
- Bigger absolute values mean stronger influence.

Why we need it:
- Random Forest is accurate but not directly interpretable.
- SHAP gives transparent feature-level evidence for the policy story.

In [ ]:
import scipy.sparse as sp
import shap

rf_b_full = Pipeline([
    ("preprocess", clone(pre_b)),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
rf_b_full.fit(X_b, y)

Xp = rf_b_full.named_steps["preprocess"].transform(X_b)
if sp.issparse(Xp):
    Xp = Xp.toarray()
Xp = np.asarray(Xp, dtype=np.float64)

rng = np.random.RandomState(42)
idx = rng.choice(Xp.shape[0], size=min(800, Xp.shape[0]), replace=False)
Xs = Xp[idx]

explainer = shap.TreeExplainer(rf_b_full.named_steps["model"])
sv = explainer.shap_values(Xs)
fn = rf_b_full.named_steps["preprocess"].get_feature_names_out()

# Global SHAP importance (numeric)
mean_abs_shap = np.abs(sv).mean(axis=0)
shap_importance_df = (
    pd.DataFrame({"feature": fn, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
)

# 1) Per-point beeswarm direction view in Plotly
top_n = 15
top_features = shap_importance_df.head(top_n)["feature"].tolist()
feat_to_rank = {f: i for i, f in enumerate(top_features)}

long_df = pd.DataFrame({
    "feature": np.repeat(fn, Xs.shape[0]),
    "shap_value": sv.T.flatten(),
    "feature_value": Xs.T.flatten(),
})
long_df = long_df[long_df["feature"].isin(top_features)].copy()
long_df["feature_rank"] = long_df["feature"].map(feat_to_rank)

# Per-feature normalization (z-score), then clip for stable color contrast
long_df["feature_value_norm"] = long_df.groupby("feature")["feature_value"].transform(
    lambda s: (s - s.mean()) / (s.std() + 1e-9)
)
long_df["feature_value_norm"] = long_df["feature_value_norm"].clip(-2.5, 2.5)

# Beeswarm jitter
jitter_rng = np.random.RandomState(42)
long_df["y_jitter"] = long_df["feature_rank"] + jitter_rng.uniform(-0.28, 0.28, size=len(long_df))

fig_swarm = px.scatter(
    long_df,
    x="shap_value",
    y="y_jitter",
    color="feature_value_norm",
    # strict blue -> pink, no white midpoint
    color_continuous_scale=[
        [0.0, "#1f6bff"],   # low values
        [1.0, "#ff4fa3"],   # high values
    ],
    range_color=[-2.5, 2.5],  # enforce consistent, vivid mapping
    opacity=0.75,
    title=f"SHAP — Model B (RF): per-point beeswarm direction view (top {top_n} features)",
    labels={
        "shap_value": "SHAP value (impact on PM2.5)",
        "feature_value_norm": "feature value (z-score)"
    },
    hover_data={
        "feature": True,
        "feature_value": ':.3f',
        "feature_value_norm": ':.3f',
        "shap_value": ':.3f',
        "y_jitter": False
    },
)

fig_swarm.update_traces(marker={"size": 6})
fig_swarm.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(
        title="",
        tickmode="array",
        tickvals=list(range(top_n)),
        ticktext=top_features,
        autorange="reversed",
    ),
)

fig_swarm.update_coloraxes(
    colorbar_title="Feature value<br>(low → high)"
)

fig_swarm.add_vline(x=0, line_dash="dash", line_color="black")
fig_swarm.write_image("images/modelb_shap_beeswarm.png", scale=2)
fig_swarm.show()


**How the color scale is formed (SHAP beeswarm)**

Colors show feature value level, not SHAP magnitude.
For each feature, values are standardized with:

$$z = \frac{x - \mu}{\sigma + 10^{-9}}$$

So:

- Blue = lower-than-average value (negative (z))
- Pink = higher-than-average value (positive (z))

This makes colors comparable across features with different units.

In [ ]:
# 2) Top 15 features in Plotly
top15 = shap_importance_df.head(15).copy().iloc[::-1]
fig_top = px.bar(
    top15,
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title="SHAP — Model B (RF): top 15 mean |SHAP|",
)
fig_top.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
fig_top.write_image("images/modelb_shap_top15.png", scale=2)
fig_top.show()


In [ ]:
# 3) Summary table + export
top10_shap = shap_importance_df.head(10).copy()
print("Top 10 SHAP features (mean |SHAP|):")
display(top10_shap)
shap_importance_df.to_csv("model_output/modelb_shap_importance.csv", index=False)
print("Saved to model_output/modelb_shap_importance.csv")

## Model C — Ridge Regression (Linear Baseline)

Train Ridge on the same Model B feature matrix, evaluate on both time and spatial splits, and report signed coefficients for interpretation.

How to read this section:
- Higher R2 and lower RMSE/MAE indicate better fit.
- Coefficient sign gives direction (positive increases predicted PM2.5, negative decreases it).
- Coefficients should be interpreted alongside SHAP because one-hot encoding and correlated variables can spread linear weights.




In [ ]:
# 1. Feature Selection: model A
features_c = features_a.copy()
X_c = df_model[features_c].copy()

print(f"Model C features: {len(features_c)}")
print(f"Model C X shape: {X_c.shape}")


### Training of model C
We train a Ridge Regression model using the same preprocessing pipeline as Model A.

The C model is evaluated on two splits:
- **Time split**: tests how well the model predicts future observations.
- **Spatial split**: tests how well the model generalizes across different locations.

Ridge is used as a linear baseline model, allowing us to compare its performance with more complex models.

In [ ]:
# 2. Training
ridge_pipe = Pipeline([
    ("preprocess", clone(pre_a)),
    ("model", Ridge(alpha=1.0, random_state=42)),
])

# Time Split
X_train_c_t = X_c.loc[time_train_mask]
X_test_c_t = X_c.loc[time_test_mask]
y_train_t = y.loc[time_train_mask]
y_test_t = y.loc[time_test_mask]

ridge_pipe.fit(X_train_c_t, y_train_t)
pred_ridge_t = ridge_pipe.predict(X_test_c_t)
m_ridge_t = reg_metrics(y_test_t, pred_ridge_t)
print("Ridge time split: ", m_ridge_t)

# Spatial Split
X_train_c_s = X_c.iloc[tr_idx]
X_test_c_s = X_c.iloc[te_idx]
y_train_s = y.iloc[tr_idx]
y_test_s = y.iloc[te_idx]

ridge_pipe_s = clone(ridge_pipe)
ridge_pipe_s.fit(X_train_c_s, y_train_s)
pred_ridge_s = ridge_pipe_s.predict(X_test_c_s)
m_ridge_s = reg_metrics(y_test_s, pred_ridge_s)
print("Ridge spatial split:", m_ridge_s)

# Metrics Table & Export
metrics_c = pd.DataFrame([
    {"split": "time",    "model": "Ridge_C", **m_ridge_t},
    {"split": "spatial", "model": "Ridge_C", **m_ridge_s},
])
metrics_c.to_csv("model_output/model_c_metrics.csv", index=False)
display(metrics_c)



#### Model C — Performance Evaluation

The Ridge model achieves:

**Time split**
  - MAE ≈ 5.21
  - RMSE ≈ 10.00
  - R² ≈ 0.35

**Spatial split**
  - MAE ≈ 4.92
  - RMSE ≈ 6.84
  - R² ≈ 0.32

The model shows moderate predictive performance, capturing some variation in PM2.5 levels, but likely missing non-linear relationships present in the data.

Performance is slightly better in the spatial split, indicating reasonable generalization across locations.

### Model C — Coefficient Analysis

To interpret the Ridge model, we retrain it on the full dataset and extract the learned coefficients.

Each coefficient represents:
- **Positive value** → increases predicted PM2.5
- **Negative value** → decreases predicted PM2.5
- **Bigger coefficients** → stronger effect

Features are ranked by their impact to identify the most important drivers.

In [ ]:
# 3. Full Retrain and Coefficient Analysis
ridge_full = clone(ridge_pipe)
ridge_full.fit(X_c, y)

feat_names_c = ridge_full.named_steps["preprocess"].get_feature_names_out()
coef_values = ridge_full.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "feature": feat_names_c,
    "coef": coef_values
})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values(by="abs_coef", ascending=False)

# Visualization
top20_coef = coef_df.head(20).sort_values("coef")
max_abs = top20_coef["coef"].abs().max()

fig_c = px.bar(
    top20_coef,
    x="coef",
    y="feature",
    orientation="h",
    color="coef",
    color_continuous_scale=[
        [0.0, "#1f6bff"],
        [1.0, "#ff4fa3"],
    ],
    title="Model C (Ridge) — Top 20 Coefficients of PM2.5",
    labels={"coef": "Coefficient Value", "feature": "Feature"}
)

fig_c.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700
)
fig_c.update_coloraxes(cmin=-max_abs, cmax=max_abs)
fig_c.add_vline(x=0, line_dash="dash", line_color="black")
fig_c.write_image("images/modelc_coefficients.png", scale=2)
fig_c.show()

In [ ]:
# Visualization
coef_df.to_csv("model_output/model_c_coefficients.csv", index=False)

print("Top 10 Positive Coefficients (Increases PM2.5)")
display(coef_df.sort_values("coef", ascending=False).head(10)[["feature", "coef"]])

print("Top 10 Negative Coefficients (Decreases PM2.5)")
display(coef_df.sort_values("coef", ascending=True).head(10)[["feature", "coef"]])

### Explanation: Top Drivers of PM2.5 (Ridge Model)

The Ridge coefficients confirm intuitive physical and spatial relationships with PM2.5:

**Top Positive Coefficients (Increase PM2.5)**

These features are associated with higher predicted PM2.5 levels.

Suburban station type, winter seasonality (`month_cos`, `Season_Winter`), lagged temperature, and higher population density are all associated with increased PM2.5 — reflecting heating emissions, reduced atmospheric mixing, and denser traffic exposure in colder months.

**Top Negative Coefficients (Decrease PM2.5)**

Altitude, current-month temperature, spring season, and wind speed consistently reduce predicted PM2.5 — capturing the dispersive effect of elevation, warmth, and airflow. Rural station classification follows the same direction.

The overall coefficient pattern is physically coherent: cold, dense, low-lying, and suburban environments drive pollution up, while warm, elevated, and well-ventilated ones bring it down.


## Model A vs Model C — Performance Comparison


In [ ]:
metrics_ac = pd.DataFrame([
    {"split": "time",    "model": f"{champion_a_name}_A", **m_champ_a_t},
    {"split": "spatial", "model": f"{champion_a_name}_A", **m_champ_a_s},
    {"split": "time",    "model": "Ridge_C", **m_ridge_t},
    {"split": "spatial", "model": "Ridge_C", **m_ridge_s},
])

display(metrics_ac)
metrics_ac.to_csv("model_output/modela_modelc_metrics_comparison.csv", index=False)

Lower RMSE and MAE indicate better predictive accuracy, while higher R² shows better explanatory power.

In [ ]:
# Visualization of metrics comparison
plot_ac = metrics_ac.melt(
    id_vars=["split", "model"],
    value_vars=["RMSE", "MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_ac,
    x="model",
    y="value",
    color="split",
    facet_col="metric",
    barmode="group",
    title=f"Model A ({champion_a_name}) vs Model C (Ridge) — RMSE and MAE by split",
    color_discrete_sequence=["#1f6bff", "#ff4fa3"],
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_image("images/modela_modelc_metrics_comparison.png", scale=2)
fig.show()

### Model A vs Model C — Time split

In [ ]:
r_a_t = y_test_t.values - pred_champ_a_t
r_c_t = y_test_t.values - pred_ridge_t

all_r_t = np.concatenate([r_a_t, r_c_t])
bins_t = np.histogram_bin_edges(all_r_t, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Model A {champion_a_name} residuals (time test)",
        "Model C Ridge residuals (time test)"
    )
)

fig.add_trace(
    go.Histogram(
        x=r_a_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name=f"{champion_a_name}_A",
        marker_color="#1f6bff",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_c_t,
        xbins=dict(start=bins_t[0], end=bins_t[-1], size=bins_t[1] - bins_t[0]),
        name="Ridge_C",
        marker_color="#ff4fa3",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title=f"Model A ({champion_a_name}) vs Model C (Ridge) residuals — time test",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)


fig.write_image("images/modela_modelc_residuals_time.png", scale=2)
fig.show()


The left histogram shows Model A (blue), while the right histogram shows Model C (pink).
Residuals are defined as actual − predicted. As we can see the Model A has more precise predictions compared to Model C on time split.

### Model A vs Model C — Spatial split

In [ ]:
r_a_s = y_test_s.values - pred_champ_a_s
r_c_s = y_test_s.values - pred_ridge_s

all_r_s = np.concatenate([r_a_s, r_c_s])
bins_s = np.histogram_bin_edges(all_r_s, bins=30)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Model A {champion_a_name} residuals (spatial test)",
        "Model C Ridge residuals (spatial test)"
    )
)

fig.add_trace(
    go.Histogram(
        x=r_a_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name=f"{champion_a_name}_A",
        marker_color="#1f6bff",
        showlegend=False
    ),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=r_c_s,
        xbins=dict(start=bins_s[0], end=bins_s[-1], size=bins_s[1] - bins_s[0]),
        name="Ridge_C",
        marker_color="#ff4fa3",
        showlegend=False
    ),
    row=1, col=2
)

fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="black", row=1, col=2)

fig.update_xaxes(title_text="actual - pred", row=1, col=1)
fig.update_xaxes(title_text="actual - pred", row=1, col=2)
fig.update_yaxes(title_text="count", row=1, col=1)
fig.update_yaxes(title_text="count", row=1, col=2)

fig.update_layout(
    title=f"Model A ({champion_a_name}) vs Model C (Ridge) residuals — spatial test",
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=1100,
    height=450
)

fig.write_image("images/modela_modelc_residuals_spatial.png", scale=2)
fig.show()

The left histogram shows Model A (RF, blue), while the right histogram shows Model C (Ridge, pink).
Residuals are defined as actual − predicted and are evaluated on the spatial test set. Predictions remain more precise for Model A on spacial split too.

In [ ]:
# Top 20 Transformed features
ridge_cmp = coef_df.copy()[["feature", "coef", "abs_coef"]]
ridge_cmp = ridge_cmp.rename(columns={"coef": "ridge_coef", "abs_coef": "abs_ridge_coef"})

shap_cmp_a = shap_importance_a.copy()

comparison_ac = (
    shap_cmp_a
    .merge(ridge_cmp, on="feature", how="inner")
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

comparison_ac["rank_shap_a"] = np.arange(1, len(comparison_ac) + 1)
comparison_ac["rank_abs_ridge"] = comparison_ac["abs_ridge_coef"].rank(
    ascending=False,
    method="dense"
).astype(int)

comparison_ac["ridge_direction"] = np.where(
    comparison_ac["ridge_coef"] >= 0,
    "increase",
    "decrease"
)

print(f"Top 20 transformed features: Model A ({champion_a_name}) SHAP vs Model C Ridge")
display(comparison_ac.head(20))

comparison_ac.to_csv("model_output/modela_modelc_feature_comparison.csv", index=False)

In [ ]:
top_n = 15
plot_ac = comparison_ac[["feature", "mean_abs_shap", "abs_ridge_coef"]].copy()

# Normalize within each model to make scales comparable
plot_ac["rf_importance_norm"] = plot_ac["mean_abs_shap"] / plot_ac["mean_abs_shap"].max()
plot_ac["ridge_importance_norm"] = plot_ac["abs_ridge_coef"] / plot_ac["abs_ridge_coef"].max()

# Average normalized importance across both models
plot_ac["avg_importance"] = (
    plot_ac["rf_importance_norm"] + plot_ac["ridge_importance_norm"]
) / 2

# Top shared features
top_ac = plot_ac.sort_values("avg_importance", ascending=False).head(top_n).copy()
top_ac = top_ac.iloc[::-1]

# Convert to long format for grouped bar chart
plot_long = top_ac.melt(
    id_vars="feature",
    value_vars=["rf_importance_norm", "ridge_importance_norm"],
    var_name="model",
    value_name="importance"
)

# Cleaner labels
plot_long["model"] = plot_long["model"].map({
    "rf_importance_norm": "Model A (RF, normalized SHAP)",
    "ridge_importance_norm": "Model C (Ridge, normalized |coef|)"
})

fig = px.bar(
    plot_long,
    x="importance",
    y="feature",
    color="model",
    barmode="group",
    orientation="h",
    title=f"Model A (RF) vs Model C (Ridge) — Top {top_n} Shared Features",
    labels={
        "importance": "Normalized importance",
        "feature": "Feature",
        "model": "Model"
    }
)

fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    xaxis_title="Normalized importance (within each model)",
    yaxis_title=""
)

fig.write_image("images/modela_modelc_feature_effect_comparison.png", scale=2)
fig.show()

Model A places the strongest emphasis on seasonality and spatial structure, with *month (cyclical encoding)*, latitude, and longitude emerging as dominant drivers. This indicates that the model captures complex interactions between time and geography.

In contrast, Model C assigns more importance to categorical and environmental variables, such as station area (suburban, urban, rural), season indicators, and altitude. This reflects the linear nature of Ridge regression, where effects are distributed across multiple variables rather than concentrated.

Notably, Model C relies much more heavily on station type and seasonal dummy variables, while Model A focuses on continuous spatial and temporal patterns.

Overall, Model A concentrates importance in a few key drivers, whereas Model C spreads importance across many features, confirming that linear models struggle to isolate the dominant non-linear factors driving PM2.5.

### Conclusion. Model A (Random Forest) vs. Model C (Ridge)

Before selecting the final model, we compared the non-linear "Policy" model (A) against the linear "Baseline" model (C) to understand the nature of the data.

| Feature | Model A (Random Forest) | Model C (Ridge Regression) |
| :--- | :--- | :--- |
| **Predictive Power** | **High:** $R^2$ of 0.29 (Time) and 0.47 (Spatial). | **Low:** $R^2$ of 0.10 (Time) and 0.28 (Spatial). |
| **Data Relationship** | Captures complex non-linear interactions. | Assumes simple linear relationships. |
| **Residuals** | Tightly centered around zero; stable. | Wider spread; tends to underestimate peaks. |
| **Top Drivers** | Month, Latitude, Longitude, Altitude. | Suburban Area, Month, Altitude, Wind Speed. |
| **Primary Use** | Policy simulation and "What-if" scenarios. | Interpretability and coefficient direction. |

**Key Takeaway:** The significant performance gap between A and C provides mathematical proof that PM2.5 levels are driven by **non-linear dynamics**. Model C underfits the data, while Model A successfully captures the complex interactions between geography and climate.


## Overall Performance Comparison

In [ ]:
# Final avg performance summary — champion A only, not both RF_A and XGBoost_A
metrics_all_final = pd.concat([
    metrics_a[metrics_a["model"] == f"{champion_a_name}_A"],
    metrics_b,
    metrics_ac[metrics_ac["model"] == "Ridge_C"],
], ignore_index=True)

avg_by_model = (
    metrics_all_final.groupby("model", as_index=False)
    .agg(avg_RMSE=("RMSE", "mean"), avg_MAE=("MAE", "mean"), avg_R2=("R2", "mean"))
    .sort_values(["avg_RMSE", "avg_MAE", "avg_R2"], ascending=[True, True, False])
)
print("Champion model by avg RMSE:", avg_by_model.iloc[0]["model"])
display(avg_by_model)
avg_by_model.to_csv("model_output/policy_model_comparison.csv", index=False)
print("Saved: model_output/all_model_comparison.csv")

In [ ]:
plot_avg = avg_by_model.melt(
    id_vars=["model"],
    value_vars=["avg_RMSE", "avg_MAE"],
    var_name="metric",
    value_name="value",
)

fig = px.bar(
    plot_avg,
    x="model",
    y="value",
    color="metric",
    barmode="group",
    title="Average Model Performance (time and spatial) — lower is better",
    labels={"value": "average value", "metric": "metric"},
    color_discrete_map={"avg_RMSE": "#1f6bff", "avg_MAE": "#ff4fa3"},
)
fig.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
)
fig.write_image("images/all_models_avg_performance.png", scale=2)
fig.show()

In [ ]:
avg_r2 = (
    metrics_all.groupby("model", as_index=False)
    .agg(avg_R2=("R2", "mean"))
    .sort_values("avg_R2", ascending=False)
)

fig_avg_r2 = px.bar(
    avg_r2,
    x="model",
    y="avg_R2",
    title="Average R² per Model (higher is better)",
    labels={"avg_R2": "average R²"},
    color="model",
)
fig_avg_r2.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    showlegend=False,
)
fig_avg_r2.write_image("images/all_models_avg_r2.png", scale=2)
fig_avg_r2.show()

### Overall Model Performance Comparison

| Model | Split | RMSE | MAE | R² |
| :--- | :--- | :--- | :--- | :--- |
| **Model A** (RF – No Lags) | Time | 10.47 | 4.38 | 0.289 |
| **Model B** (RF – With Lags) | Time | 10.46 | 4.31 | 0.290 |
| **Model C** (Ridge) | Time | 11.77 | 6.04 | 0.101 |
| **Model A** (RF – No Lags) | Spatial | 6.00 | 3.86 | 0.474 |
| **Model B** (RF – With Lags) | Spatial | **4.75** | **2.90** | **0.671** |
| **Model C** (Ridge) | Spatial | 7.02 | 4.90 | 0.282 |

---

### Final Model Recommendation

**The Power of Persistence (Model B)**
Model B is the most robust predictor, achieving a "massive leap" in the **Spatial split ($R^2 = 0.671$)**. This indicates that historical PM2.5 data (lags) allows the model to "anchor" its predictions to a specific location's baseline. It confirms that pollution is a process of **temporal persistence**—today's air quality is heavily influenced by the accumulation of previous days.

**Proposed Deployment Strategy:**
We recommend a dual-model approach based on the project objective:

1.  **For High-Accuracy Forecasting (Model B):** This is the preferred choice for operational deployment. Its superior ability to generalize to unseen regions makes it the most reliable tool for real-time monitoring.
2.  **For Urban Planning & Policy (Model A):** While slightly less accurate, Model A is the primary tool for simulation. Because it does not rely on previous pollution levels (which a city cannot change), it allows us to isolate and simulate how changing actionable variables—like green ratios or traffic features—would directly impact air quality.

**Conclusion:** We suggest to use **Model B** to predict future levels with high precision; use **Model A** to understand and simulate the impact of structural environmental changes.

## Policy Translation
This section takes the combined model outputs and translates them into actionable policy insights.

Features that ranked highly in **both** Model A (SHAP importance) and Model C (Ridge coefficient magnitude) are treated as the most robust signals — confirmed by two independent modelling approaches from different angles. A `cross_model_strength` score is computed as the product of mean absolute SHAP value and absolute Ridge coefficient, surfacing the features where both models agree on importance.

Each feature is then mapped to a recommended policy action based on its direction and domain context.

In [ ]:
# Feature signals — features that matter in both Model A (SHAP) and Model C (Ridge)

signals = comparison_ac.copy()

# Model A SHAP importance * by absolute Ridge coefficient (Model C)
# we ignore direction (positive/negative) and focus on magnitude
signals["cross_model_strength"] = signals["mean_abs_shap"] * signals["abs_ridge_coef"]
signals = signals.sort_values("cross_model_strength", ascending=False) # sorting

policy_signals = signals.head(20).copy()
display(policy_signals)
policy_signals.to_csv("model_output/policy_feature_signals.csv", index=False)
print("Saved: model_output/policy_feature_signals.csv")

In [ ]:
# Proposed actions per feature
def action_for_feature(f):
    f_low = str(f).lower()
    if "green_ratio" in f_low:
        return "Increase urban greening and tree canopy in dense neighborhoods"
    if "wind" in f_low:
        return "Use ventilation corridors in urban planning"
    if "precipitation" in f_low:
        return "Plan seasonal mitigation for dry periods"
    if "temp" in f_low or "month" in f_low:
        return "Target seasonal campaigns and winter anti-inversion policies"
    if "no2" in f_low or "pm10" in f_low:
        return "Prioritize traffic/combustion emission reduction in hotspots"
    if "pm2_5_lag" in f_low or "roll3" in f_low:
        return "Deploy early warning when persistence is high"
    if "latitude" in f_low or "longitude" in f_low or "altitude" in f_low:
        return "Use place-based local plans by geography"
    if "population" in f_low:
        return "Target high-density urban areas for intervention"
    if "station_type" in f_low or "station_area" in f_low:
        return "Apply zone-specific regulation (traffic vs background areas)"
    return "Investigate targeted local intervention"

actions = policy_signals[["feature", "mean_abs_shap", "ridge_coef", "ridge_direction"]].copy()
actions["recommended_action"] = actions["feature"].apply(action_for_feature)
display(actions.head(15))
actions.to_csv("model_output/policy_actions.csv", index=False)
print("Saved: model_output/policy_actions.csv")

#### Results

The cross-model signals converge on a consistent set of policy levers:

- **Seasonality** (`month_cos`, `Season_Winter`, `Temp_Mean`) is the dominant driver across both models. The clearest policy implication is targeted winter intervention — anti-inversion campaigns, heating emission controls, and seasonal traffic restrictions during cold months.
- **Geography** (`Latitude`, `Longitude`, `Altitude`) ranks among the strongest signals, indicating that place-based planning is essential. Municipalities in lower-altitude, northern, and more eastern areas face structurally higher PM2.5 regardless of local policy — regional coordination is needed.
- **Urban structure** (`Station_Area_Suburban`, `Population_Density`, `Station_Type_Traffic`) points to zone-specific regulation: suburban and traffic-exposed areas require different intervention strategies than rural or background stations.
- **Wind and precipitation** confirm the role of atmospheric dispersion as a natural buffer — urban planning that preserves ventilation corridors and greening can reduce baseline exposure.

The resulting `policy_actions.csv` provides a feature-level action table ready for stakeholder reporting, combining model-derived importance scores with directional evidence and concrete intervention categories.